In [1]:
import numpy as np
import pandas as pd
import pickle
import networkx as nx

INPUT = "code/dataset/Original/global/global-genre_network-2019.csv"
df = pd.read_csv(INPUT, sep="\t")
df["weight"] = df["weight"].astype(float)

genres = pd.unique(df[["source", "target"]].values.ravel())
genre_to_idx = {g: i for i, g in enumerate(genres)}
n = len(genres)

i = df["source"].map(genre_to_idx).to_numpy()
j = df["target"].map(genre_to_idx).to_numpy()
w = df["weight"].to_numpy()

adj_numpy = np.zeros((n, n), dtype=np.float32)

for a, b, val in zip(i, j, w):
    adj_numpy[a, b] = max(adj_numpy[a, b], val)
    adj_numpy[b, a] = max(adj_numpy[b, a], val)

np.fill_diagonal(adj_numpy, 0)

print("Original shape:", adj_numpy.shape)

G = nx.from_numpy_array(adj_numpy)
components = list(nx.connected_components(G))
print("Number of connected components:", len(components))

largest_cc = max(components, key=len)
largest_cc = sorted(largest_cc)
print("Largest CC size:", len(largest_cc))

adj_lcc = adj_numpy[np.ix_(largest_cc, largest_cc)]
genres_lcc = [genres[idx] for idx in largest_cc]

G_lcc = nx.from_numpy_array(adj_lcc)
print("LCC shape:", adj_lcc.shape)
print("LCC connected:", nx.is_connected(G_lcc))

# Remove self-loops on LCC slice
np.fill_diagonal(adj_lcc, 0)

# Create directed version using degree-based asymmetry
degree = adj_lcc.sum(axis=1)

adj_directed = np.zeros_like(adj_lcc)
for i in range(len(adj_lcc)):
    for j in range(len(adj_lcc)):
        if adj_lcc[i, j] > 0 and degree[i] > 0:
            adj_directed[i, j] = adj_lcc[i, j] * (degree[j] / degree[i])

# Normalize rows
row_sums = adj_directed.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
adj_lcc = adj_directed / row_sums

print("Diagonal sum after cleanup:", adj_lcc.diagonal().sum())
print("Matrix is symmetric:", np.allclose(adj_lcc, adj_lcc.T))  # should be False now

np.save("my_graph.npy", adj_lcc)
pickle.dump(genres_lcc, open("matrix_vocab.pkl", "wb"))

Original shape: (310, 310)
Number of connected components: 4
Largest CC size: 294
LCC shape: (294, 294)
LCC connected: True
Diagonal sum after cleanup: 0.0
Matrix is symmetric: False


In [2]:
print("Largest CC size:", len(largest_cc))

Largest CC size: 294
